# Lecture 3 — Class Exercise
## Line Charts & Slopegraphs: CO2 Emissions

> **Push to:** `week03/lecture03_exercise.ipynb` in your GitHub repo

### Remember:
1. No spaghetti — multiple lines must use grey + single highlight
2. Remove clutter: no chart borders, no heavy gridlines, no legend if you can label directly
3. Insight title — states the finding, not the topic
4. Carry forward from Lecture 2: white background, Arial font, professional quality


In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Dataset: CO2 Emissions by Country 2000-2022
# Source: Our World in Data (https://ourworldindata.org/co2-emissions)
df = pd.read_csv('/content/co2_emissions.csv')
print(f"Loaded: {len(df)} rows | Countries: {df['Country'].nunique()} | Years: {df['Year'].min()}-{df['Year'].max()}")
print(df.head())


Loaded: 345 rows | Countries: 15 | Years: 2000-2022
         Country         Region  Year  CO2_Mt  CO2_per_capita
0  United States  North America  2000  5857.6            1.32
1  United States  North America  2001  5724.0            1.26
2  United States  North America  2002  5652.8            1.11
3  United States  North America  2003  5592.8            1.29
4  United States  North America  2004  5743.2            1.12


In [2]:
# Explore before building

print("Countries:", df['Country'].unique())
print("\nCO2 range:", df['CO2_Mt'].min(), "to", df['CO2_Mt'].max(), "Mt")
print("\nRegional averages (2022):")
print(df[df['Year']==2022].groupby('Region')['CO2_Mt'].mean().sort_values(ascending=False).round(1))


Countries: ['United States' 'China' 'India' 'Germany' 'United Kingdom' 'France'
 'Brazil' 'Japan' 'Canada' 'Australia' 'South Korea' 'Russia'
 'South Africa' 'Mexico' 'Indonesia']

CO2 range: 125.3 to 12409.5 Mt

Regional averages (2022):
Region
Asia             3531.1
North America    2393.8
Latin America     629.2
Africa            534.4
Europe            496.5
Oceania           493.7
Name: CO2_Mt, dtype: float64


---
## Task 1 — Multi-Series Line Chart with Highlight

**What to build:** A line chart showing CO2 emissions over time for **all Asian countries** in the dataset, with one country highlighted.

**Requirements:**
- All countries shown (for context), but only **one highlighted in colour** — your choice which
- All other lines in grey (#DDDDDD), thinner
- Highlighted country **labelled directly** at the end of its line (not in a legend)
- Insight title that names the highlighted country and its story

> 💡 `df[df['Region'] == 'Asia']` to filter; use `go.Figure()` with a loop for per-country control


In [3]:
# Task 1 — Multi-series line with highlight
# YOUR CODE HERE
import pandas as pd
import plotly.graph_objects as go

# Load dataset
df = pd.read_csv("co2_emissions.csv")

# Filter for Asia
asia_df = df[df['Region'] == 'Asia']

# Highlight country
highlight_country = "India"

# Create figure
fig = go.Figure()

# Loop through countries
for country in asia_df['Country'].unique():
    country_df = asia_df[asia_df['Country'] == country]

    if country == highlight_country:
        # Highlighted line (India)
        fig.add_trace(go.Scatter(
            x=country_df['Year'],
            y=country_df['CO2_Mt'],
            mode='lines',
            line=dict(color='#E63946', width=3),
            showlegend=False
        ))

        # Direct label at end
        fig.add_trace(go.Scatter(
            x=[country_df['Year'].iloc[-1]],
            y=[country_df['CO2_Mt'].iloc[-1]],
            text=[highlight_country],
            mode='text',
            textposition='middle right',
            showlegend=False
        ))

    else:
        # Background countries
        fig.add_trace(go.Scatter(
            x=country_df['Year'],
            y=country_df['CO2_Mt'],
            mode='lines',
            line=dict(color='#DDDDDD', width=1),
            showlegend=False
        ))

# Layout with insight title
fig.update_layout(
    title="India’s total CO₂ emissions have risen sharply over time, standing out among Asian countries",
    xaxis_title="Year",
    yaxis_title="CO₂ Emissions (Mt)",
    template="simple_white"
)

fig.show()

---
## Task 2 — Slopegraph: Regional Change 2000 vs 2022

**What to build:** A slopegraph comparing **average regional CO2 emissions** between 2000 and 2022.

**Requirements:**
- One line per region (not per country — aggregate first)
- Colour: regions that increased = one colour; decreased = another
- Values labelled at both ends of each line
- No y-axis tick labels (the endpoint labels make them redundant)
- Insight title stating which regions moved most

> 💡 `df.groupby(['Region','Year'])['CO2_Mt'].mean().reset_index()` then filter to 2000 and 2022


In [4]:
# Task 2 — Slopegraph: regional averages
# YOUR CODE HERE

import pandas as pd
import plotly.graph_objects as go

# Load dataset
df = pd.read_csv("co2_emissions.csv")

# Aggregate: average CO2 emissions per region per year
regional_avg = df.groupby(['Region', 'Year'])['CO2_Mt'].mean().reset_index()

# Filter for 2000 and 2022
slope_df = regional_avg[regional_avg['Year'].isin([2000, 2022])]

# Pivot for easier plotting
pivot_df = slope_df.pivot(index='Region', columns='Year', values='CO2_Mt').reset_index()

# Create figure
fig = go.Figure()

# Loop through each region
for _, row in pivot_df.iterrows():
    region = row['Region']
    y_start = row[2000]
    y_end = row[2022]

    # Color based on increase or decrease
    color = '#E63946' if y_end > y_start else '#457B9D'

    # Line
    fig.add_trace(go.Scatter(
        x=[2000, 2022],
        y=[y_start, y_end],
        mode='lines',
        line=dict(color=color, width=2),
        showlegend=False
    ))

    # Label at start (2000)
    fig.add_trace(go.Scatter(
        x=[2000],
        y=[y_start],
        text=[f"{region} ({y_start:.1f})"],
        mode='text',
        textposition='middle left',
        showlegend=False
    ))

    # Label at end (2022)
    fig.add_trace(go.Scatter(
        x=[2022],
        y=[y_end],
        text=[f"{region} ({y_end:.1f})"],
        mode='text',
        textposition='middle right',
        showlegend=False
    ))

# Layout styling
fig.update_layout(
    title="Most regions increased average CO₂ emissions from 2000 to 2022, with the largest rises in rapidly developing areas",
    xaxis=dict(
        tickvals=[2000, 2022],
        title=""
    ),
    yaxis=dict(
        showticklabels=False,
        title="Average CO₂ Emissions (Mt)"
    ),
    template="simple_white"
)

fig.show()